# TR Synthetic Ticket Generation on Kaggle (Ollama + Qwen3.5:9b)

**Before running:** Settings (top right) → Accelerator → **GPU T4 x2**

Steps:
1. Install Ollama
2. Start Ollama server in background
3. Pull qwen3.5:9b model
4. Upload project files
5. Quick quality test
6. Run full generation (resumable)
7. Download result

## 1. Install Ollama

In [2]:
!apt-get update -qq && apt-get install -y -qq zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## 2. Start Ollama server in background

In [4]:
import subprocess, os, time

subprocess.Popen(
    ['nohup', 'ollama', 'serve'],
    stdout=open('ollama.log', 'w'),
    stderr=open('ollama_error.log', 'w'),
    preexec_fn=os.setsid,
)
print('Ollama server starting...')
time.sleep(5)
!curl -s http://localhost:11434 || echo 'Not ready yet, wait a few seconds and re-run'

Ollama server starting...
Ollama is running

## 3. Check GPU and pull model

Kaggle gives T4 x2 (32GB total VRAM) — qwen3.5:9b fits easily.

In [5]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 14912 MiB
Tesla T4, 15360 MiB, 14912 MiB


In [6]:
!ollama pull qwen3.5:9b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling dec52a44569a:   0% ▕                  ▏ 1.0 MB/6.6 GB                  pulling manifest 
pulling dec52a44569a:   1% ▕                  ▏  52 MB/6.6 GB                  pulling manifest 
pulling dec52a44569a:   2% ▕                  ▏ 142 MB/6.6 GB                  pulling manifest 
pulling dec52a44569a:   3% ▕                  ▏ 186 MB/6.6 GB                  pulling manifest 
pulling dec52a44569a:   4% ▕                  ▏ 277 MB/6.6 GB                  pulling manifest 
pulling dec52a44569a:   6% ▕█                 ▏ 375 MB/6.6 GB                  pulling manifest 
pulling dec52a44569a:   6% ▕█                 ▏ 419 MB/6.6 GB                  pulling manifest 
pulling dec52a44569a:   8% ▕█                 ▏ 510 MB/6.6 GB                  pulling manifest 
pulling dec52a44569a:   9% 

## 4. Upload project files

Upload these files using the cell below:
- `schema.py`
- `generate_skeletons.py`
- `expand_with_ollama.py`
- `skeletons_full.jsonl`

**Important:** Kaggle does not have a built-in file upload widget like Colab.
Instead, use the **+ Add Data** button on the right panel, or simply copy-paste
the files to /kaggle/working/ using the cell below.

In [11]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/mustafarikan/testdt-1/skeletons_full.jsonl
/kaggle/input/datasets/mustafarikan/testdt-1/tickets_tr_full.jsonl
/kaggle/input/datasets/mustafarikan/testdt-1/generate_skeletons.py
/kaggle/input/datasets/mustafarikan/testdt-1/schema.py
/kaggle/input/datasets/mustafarikan/testdt-1/expand_with_ollama.py


In [12]:
# Kaggle does not have files.upload() like Colab.
# Add your files as a Kaggle Dataset (recommended) or use kaggle API.
# Alternatively, paste file contents directly here.
#
# Easiest approach: in the right panel click '+ Add Data' → 'Upload' → drag your files.
# They will appear under /kaggle/input/YOUR-DATASET-NAME/
# Then copy them to working directory:

import shutil, os

# Change 'your-dataset-name' to the actual name you gave your dataset
DATASET_PATH = '/kaggle/input/datasets/mustafarikan/testdt-1'
if os.path.exists(DATASET_PATH):
    for f in os.listdir(DATASET_PATH):
        shutil.copy(os.path.join(DATASET_PATH, f), f'/kaggle/working/{f}')
        print(f'Copied: {f}')
else:
    print(f'Dataset not found at {DATASET_PATH}')
    print('Please add your dataset via the right panel (+Add Data) first.')

!ls -la /kaggle/working/

Copied: skeletons_full.jsonl
Copied: tickets_tr_full.jsonl
Copied: generate_skeletons.py
Copied: schema.py
Copied: expand_with_ollama.py
total 4436
drwxr-xr-x 3 root root    4096 Jul  3 05:09 .
drwxr-xr-x 5 root root    4096 Jul  3 05:05 ..
-rw-r--r-- 1 root root   11135 Jul  3 05:14 expand_with_ollama.py
-rw-r--r-- 1 root root   11857 Jul  3 05:14 generate_skeletons.py
-rw-r--r-- 1 root root   22248 Jul  3 05:10 ollama_error.log
-rw-r--r-- 1 root root     612 Jul  3 05:10 ollama.log
-rw-r--r-- 1 root root    8488 Jul  3 05:14 schema.py
-rw-r--r-- 1 root root 1484585 Jul  3 05:14 skeletons_full.jsonl
-rw-r--r-- 1 root root 2975667 Jul  3 05:14 tickets_tr_full.jsonl
drwxr-xr-x 2 root root    4096 Jul  3 05:05 .virtual_documents


## 5. Quick quality test (10 rows)

In [9]:
!nvidia-smi

Fri Jul  3 05:09:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
with open("/kaggle/working/tickets_tr_full.jsonl") as f:
    lines = f.readlines()
print(f"Toplam satır: {len(lines)}")

Toplam satır: 2961


In [10]:
import subprocess
result = subprocess.run(
    ['python', '/kaggle/working/expand_with_ollama.py',
     '--in', '/kaggle/working/skeletons_full.jsonl',
     '--out', '/kaggle/working/tickets_tr_full.jsonl',
     '--model', 'qwen3.5:9b',
     '--limit', '10'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:500])

KeyboardInterrupt: 

## 6. Full generation run

This is the main cell. It is **resumable** — if interrupted, re-run the same cell
and it will skip already-completed rows.

**To keep the session alive:** Kaggle times out on inactivity.
Click somewhere in the notebook every 30-40 minutes, or use the
keep-alive trick below.

In [15]:
# Keep-alive: runs a harmless loop that prints a dot every 20 minutes
# to prevent idle timeout. Run this cell FIRST, then run cell 7.
import threading, time

def keep_alive():
    while True:
        time.sleep(1200)  # 20 minutes
        print('.', end='', flush=True)

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print('Keep-alive thread started.')

Keep-alive thread started.


In [16]:
import subprocess, os, time, requests

# Ollama'yı yeniden başlat
subprocess.Popen(
    ['ollama', 'serve'],
    stdout=open('ollama.log', 'w'),
    stderr=open('ollama_error.log', 'w'),
    preexec_fn=os.setsid,
)
time.sleep(5)

# GPU'ya yükle
subprocess.Popen(['ollama', 'run', 'qwen3.5:9b', '--keepalive', '24h'],
    stdout=open('model.log', 'w'),
    stderr=open('model_error.log', 'w'),
    preexec_fn=os.setsid,
)
time.sleep(10)

# Kontrol
result = subprocess.run(['ollama', 'ps'], capture_output=True, text=True)
print(result.stdout)

NAME    ID    SIZE    PROCESSOR    CONTEXT    UNTIL 



In [17]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/mustafarikan/testdt-1/skeletons_full.jsonl
/kaggle/input/datasets/mustafarikan/testdt-1/tickets_tr_full.jsonl
/kaggle/input/datasets/mustafarikan/testdt-1/generate_skeletons.py
/kaggle/input/datasets/mustafarikan/testdt-1/schema.py
/kaggle/input/datasets/mustafarikan/testdt-1/expand_with_ollama.py


In [18]:
!python /kaggle/working/expand_with_ollama.py --in /kaggle/working/skeletons_full.jsonl --out /kaggle/working/tickets_tr_full.jsonl --model qwen3.5:9b --workers 4

Model:                    qwen3.5:9b
Workers:                  4
Total rows in input:      4225
Already completed:        2959
Remaining to process:     1266
  [1/1266] ok (70s elapsed, 32.2 tok/sec, ETA ~24.7h)
  [2/1266] ok (79s elapsed, 32.1 tok/sec, ETA ~13.9h)
  [3/1266] ok (86s elapsed, 32.0 tok/sec, ETA ~10.1h)
  [4/1266] ok (93s elapsed, 31.8 tok/sec, ETA ~8.2h)
  [5/1266] ok (104s elapsed, 31.6 tok/sec, ETA ~7.3h)
  [10/1266] ok (153s elapsed, 29.3 tok/sec, ETA ~5.4h)
  [20/1266] ok (248s elapsed, 27.0 tok/sec, ETA ~4.3h)
  [30/1266] ok (343s elapsed, 26.3 tok/sec, ETA ~3.9h)
  [40/1266] ok (435s elapsed, 25.9 tok/sec, ETA ~3.7h)
  [50/1266] ok (527s elapsed, 25.7 tok/sec, ETA ~3.6h)
  [60/1266] ok (627s elapsed, 25.5 tok/sec, ETA ~3.5h)
  [70/1266] ok (734s elapsed, 25.4 tok/sec, ETA ~3.5h)
  [80/1266] ok (821s elapsed, 25.4 tok/sec, ETA ~3.4h)
  [90/1266] ok (910s elapsed, 25.3 tok/sec, ETA ~3.3h)
  [100/1266] ok (1010s elapsed, 25.2 tok/sec, ETA ~3.3h)
  [110/1266] ok (1100

In [ ]:
import subprocess, time
result = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.free', '--format=csv'], capture_output=True, text=True)
print(result.stdout)

## 7. Verify and download

In [ ]:
import json

with open('/kaggle/working/tickets_tr_full.jsonl', encoding='utf-8') as f:
    rows = [json.loads(l) for l in f if l.strip()]

errors = sum(1 for r in rows if 'error' in r)
print(f'Total rows:    {len(rows)}')
print(f'Errors:        {errors}')
print(f'Unique IDs:    {len(set(r["id"] for r in rows))}')
print()

# HTML/tag leftover check
import re
html_hits = sum(1 for r in rows if 'error' not in r
                and re.search(r'<br>|<product>|</product>', r['subject']+r['body']+r['answer']))
print(f'HTML/tag leftovers: {html_hits} ({html_hits/max(len(rows),1)*100:.1f}%)')

# 3 random samples
import random
random.seed(1)
for r in random.sample([r for r in rows if 'error' not in r], min(3, len(rows))):
    print('-'*60)
    print(f'queue: {r["queue"]} | priority: {r["priority"]} | type: {r["type"]}')
    print('SUBJECT:', r['subject'])
    print('BODY:', r['body'][:200])
    print('ANSWER:', r['answer'][:200])

In [ ]:
# Download: in Kaggle, go to the Output panel on the right
# and download tickets_tr_full.jsonl from /kaggle/working/
# Or run this to confirm the file is ready:
!ls -lh /kaggle/working/tickets_tr_full.jsonl